# Week 2 — ML Task Framing

## 1. My Lane as an ML Task

My chosen lane is **Refresh / Content Opportunity Scoring**.

I would frame this primarily as a **ranking problem**. The goal is not simply to classify every page as good or bad. Instead, I want to rank pages so that a content or SEO team can review the most important pages first.

The unit of analysis is a content page. For each page, I can use information such as impressions, CTR, average position, content age, word count, and trend-related signals to help estimate its priority for review.

The output would be a ranked list of pages. The highest-ranked pages would be the pages that deserve human attention first.


## 2. Target or Proxy

The final target will need to represent whether a page is worth prioritizing for review. At this early stage, I would use a **proxy target** based on observable page performance rather than claiming that the model can directly predict whether a specific content change will succeed.

One possible proxy is whether a page shows a meaningful downward trend in its search performance. The starter dataset contains the `trend_direction` field, which can be used to explore this idea.

For example, I could create a binary target such as:

- `1` = page is showing a downward trend and may deserve review
- `0` = page is not showing a downward trend

However, I would not automatically treat every downward-trending page as a guaranteed refresh opportunity. The target is only a starting proxy for prioritization. Later, I would investigate whether a better target can represent the actual business decision.


## 3. Success Metric

My main success metric would be **Precision@K**, such as Precision@50.

This metric makes sense because the practical goal is to give a team a short list of pages to review. If the model ranks 50 pages at the top, Precision@50 tells me how many of those top 50 pages match the target condition.

This is more useful for my problem than simply looking at overall accuracy. A model could have high accuracy while still failing to put the most useful pages near the top of the review queue.

I would also consider other ranking metrics later, but Precision@K is a useful starting point because it directly connects model performance with the limited number of pages a human team can realistically review.


In [ ]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("Dataset shape:", df.shape)

lane_df = df[
    [
        "trend_direction",
        "search_volume",
        "impressions_90d",
        "ctr",
        "avg_position",
        "content_age_days",
        "word_count"
    ]
].copy()

lane_df.head(10)


### Unit of Analysis

The unit of analysis is **one content page**.

Each row represents one page/content item, while the columns describe different characteristics and search-performance signals for that page.

For example, `impressions_90d` represents the page's impressions over the 90-day period, `ctr` represents click-through rate, `avg_position` represents its average search position, and `content_age_days` represents the age of the content.

This matters because the model will make a recommendation at the page level. The output should therefore allow a human reviewer to identify individual pages that deserve attention.


In [ ]:
lane_df["target_proxy"] = (
    lane_df["trend_direction"] == "down"
).astype(int)

print("Target distribution:")
print(lane_df["target_proxy"].value_counts())

lane_df[
    [
        "trend_direction",
        "impressions_90d",
        "ctr",
        "avg_position",
        "target_proxy"
    ]
].head(10)


### Target Sketch

I created a preliminary `target_proxy` column to show what the target could look like.

A value of `1` represents a page with a downward trend, while `0` represents a page without a downward trend.

This is only a prototype target. Before using it for a final model, I would need to confirm that this definition actually represents the decision I want to support and that there is no information leakage between the features and target.


## 5. Why ML Beats a Fixed Rule

A fixed rule could be useful as a baseline. For example, I could manually say that every page with a large drop in performance should be reviewed first.

The problem is that a single rule may ignore combinations of signals. A page's priority may depend on several factors at the same time, such as impressions, CTR, position, content age, and other characteristics.

Machine learning can learn patterns from multiple features instead of relying on one manually chosen threshold. It can then rank pages based on patterns learned from historical examples.

This does not mean ML is automatically better. A fixed rule is easier to understand and maintain, so I would keep a rule-based baseline and compare it against the ML approach.

The reason I think ML is worth testing here is that the task involves many pages and multiple interacting signals, while the final goal is to prioritize a limited number of pages for human review.


### What the Output Supports

The output of the model would be a ranked list of pages.

A content or SEO team could use this list to decide which pages to inspect first. A high-ranked page would not automatically be changed. Instead, a human would review the page and decide whether it needs a refresh, improvement, monitoring, protection, or no action.

This makes the model a decision-support tool rather than an automatic content-editing system.


## 6. Self-Check

- [x] I identified my lane as Refresh / Content Opportunity Scoring.
- [x] I identified the ML task type as ranking.
- [x] I described a possible target/proxy.
- [x] I identified Precision@K as the main success metric.
- [x] I showed the unit of analysis using a real dataframe.
- [x] I created a preliminary target column.
- [x] I explained why ML is worth testing against a fixed rule.
- [x] I connected the model output to a real content action.
- [x] I explained that the target is provisional and requires further validation.
- [x] I avoided claiming that the model proves causation.
